# Task 2 — Baseline + Multi Ensemble

두 CSV를 `[1000,1000]` bool mask로 복원한 뒤 pixel-level OR ensemble을 수행하고,
다시 RLE로 인코딩하여 최종 submission CSV를 생성합니다.


In [ ]:
# 선행 조건: 01, 03 실행으로 생성되는 csv 2개
# 결합 방식: 두 마스크의 합집합(OR)

import numpy as np
import pandas as pd

BASELINE_CSV = "./submission_task2_baseline.csv"
MULTI_CSV = "./submission_task2_multi_specialist.csv"

OUTPUT_CSV = "./submission_task2_ensemble_base_multi.csv"

H = W = 1000


In [ ]:
def pick_label_column(df):
    candidates = [
        "Label",
        "label",
        "EncodedPixels",
        "encoded_pixels",
        "mask",
    ]

    lower = {
        str(c).lower(): c
        for c in df.columns
    }

    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]

    raise KeyError(
        f"Label column not found: {list(df.columns)}"
    )

def decode_single_mask(rle, h=1000, w=1000):
    pixels = np.zeros(
        h * w,
        dtype=bool,
    )

    if pd.isna(rle):
        return pixels.reshape((w, h)).T

    rle = str(rle).strip()

    if rle == "":
        return pixels.reshape((w, h)).T

    nums = np.fromstring(
        rle,
        sep=" ",
        dtype=np.int64,
    )

    starts = nums[0::2] - 1
    lengths = nums[1::2]

    for start, length in zip(
        starts,
        lengths,
    ):
        pixels[start:start + length] = True

    return pixels.reshape((w, h)).T

# RLE 축 전치 필수 — mask.T.flatten() 기준
def encode_single_mask(mask):
    mask = np.asarray(
        mask,
        dtype=bool,
    )

    assert mask.shape == (1000, 1000)

    pixels = mask.T.flatten()

    pixels = np.concatenate([
        [False],
        pixels,
        [False],
    ])

    runs = np.where(
        pixels[1:] != pixels[:-1]
    )[0] + 1

    runs[1::2] -= runs[::2]

    return " ".join(
        str(x)
        for x in runs
    )


In [ ]:
base_df = pd.read_csv(
    BASELINE_CSV
)

multi_df = pd.read_csv(
    MULTI_CSV
)

assert len(base_df) == len(multi_df)

LABEL_COL = pick_label_column(
    base_df
)

assert LABEL_COL in multi_df.columns


In [ ]:
final_rles = []

for i in range(len(base_df)):

    base_mask = decode_single_mask(
        base_df.iloc[i][LABEL_COL]
    )

    multi_mask = decode_single_mask(
        multi_df.iloc[i][LABEL_COL]
    )

    final_mask = (
        base_mask | multi_mask
    )

    final_rles.append(
        encode_single_mask(
            final_mask
        )
    )


In [ ]:
submission = base_df.copy()

submission[LABEL_COL] = final_rles

assert len(submission) == 200
assert not submission[LABEL_COL].isna().any()

submission.to_csv(
    OUTPUT_CSV,
    index=False,
)
